In [ ]:
!pip install pykeen -q

In [ ]:
import torch
import pandas as pd
import numpy as np
import itertools
from pykeen.pipeline import pipeline
from pykeen.datasets import Nations

In [ ]:

models = ["ConvKB"]
seeds  = [42, 123, 999]
epochs = [100]

results = {}

for model, seed, epoch in itertools.product(models, seeds, epochs):
    key = f"{model}_s{seed}_e{epoch}"
    print(f"Running {key}...")
    
    results[key] = pipeline(
        dataset="Nations",
        model=model,
        training_kwargs=dict(num_epochs=epoch),
        random_seed=seed,
        use_testing_data=True,
    )
    print(f"  done. MRR={results[key].get_metric('mrr'):.4f}")

In [ ]:
rows = []

for key, result in results.items():
    model, seed_str, epoch_str = key.split("_")
    seed  = int(seed_str[1:])
    epoch = int(epoch_str[1:])
    
    mrr     = result.get_metric("mrr")
    hits1   = result.get_metric("hits_at_1")
    hits10  = result.get_metric("hits_at_10")
    
    rows.append({
        "key":    key,
        "model":  model,
        "seed":   seed,
        "epochs": epoch,
        "MRR":    round(mrr,   4),
        "H@1":    round(hits1, 4),
        "H@10":   round(hits10,4),
    })

df_ablation = pd.DataFrame(rows)
print(df_ablation.sort_values("MRR", ascending=False).to_string(index=False))

In [ ]:
from pykeen.datasets import Nations
import pandas as pd
dataset = Nations()
df = pd.DataFrame(dataset.training.triples, columns=["head","relation","tail"])
print(sorted(df["relation"].unique()))
print("\n--- triples containing usa ---")
print(df[df["head"]=="usa"].to_string(index=False))

In [ ]:
# check test split
df_test = pd.DataFrame(
    dataset.testing.triples, 
    columns=["head","relation","tail"]
)

# find usa triples in test
print(df_test[df_test["head"]=="usa"].to_string(index=False))

In [ ]:
# pick your probe query - one triple to analyze deeply
# we hide this and ask: where does the model rank the correct tail?

dataset   = Nations()
entity_to_id   = dataset.training.entity_to_id
relation_to_id = dataset.training.relation_to_id
id_to_entity   = {v: k for k, v in entity_to_id.items()}

probe_head     = "usa"
probe_relation = "independence"   
probe_tail     = "uk"

def get_failure_vector(result, head, relation, tail):
    """
    For a given trained model and probe triple,
    return the failure vector: predicted_tail_vec - true_tail_vec
    This shows the direction and magnitude of the miss.
    """
    model = result.model
    
    h_id = entity_to_id[head]
    r_id = relation_to_id[relation]
    t_id = entity_to_id[tail]
    
    # get entity embeddings
    entity_embs = model.entity_representations[0](indices=None).detach()
    
    # get the predicted tail: for TransH/DistMult this is the 
    # highest scoring entity given (h, r)
    h_vec = entity_embs[h_id]
    t_vec = entity_embs[t_id]
    
    # score all entities as candidate tails
    scores = []
    for i in range(len(entity_to_id)):
        candidate = entity_embs[i]
        # use model's built-in score function
        triple_tensor = torch.tensor([[h_id, r_id, i]])
        score = model.score_hrt(triple_tensor).item()
        scores.append((i, score))
    
    # rank them
    scores.sort(key=lambda x: -x[1])
    ranked_ids = [s[0] for s in scores]
    
    rank = ranked_ids.index(t_id) + 1  # 1-indexed
    predicted_tail_id = ranked_ids[0]
    
    # failure vector = where model pointed - where it should have pointed
    predicted_vec = entity_embs[predicted_tail_id].cpu().numpy()
    true_vec      = t_vec.cpu().numpy()
    failure_vec   = predicted_vec - true_vec
    
    return {
        "rank":          rank,
        "reciprocal":    1/rank,
        "predicted":     id_to_entity[predicted_tail_id],
        "failure_vec":   failure_vec,
        "failure_mag":   float(np.linalg.norm(failure_vec)),
    }

# compute failure vectors across seeds for each model
stability_rows = []

for model_name in models:
    for epoch in epochs:
        vecs = []
        for seed in seeds:
            key    = f"{model_name}_s{seed}_e{epoch}"
            result = results[key]
            info   = get_failure_vector(
                result, probe_head, probe_relation, probe_tail
            )
            vecs.append(info["failure_vec"])
            print(f"{key}: rank={info['rank']}, "
                  f"predicted='{info['predicted']}', "
                  f"magnitude={info['failure_mag']:.3f}")
        
        # stability = average pairwise cosine similarity across seeds
        sims = []
        for i in range(len(vecs)):
            for j in range(i+1, len(vecs)):
                a, b = vecs[i], vecs[j]
                cos  = np.dot(a,b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-8)
                sims.append(cos)
        
        stability = float(np.mean(sims))
        stability_rows.append({
            "model":     model_name,
            "epochs":    epoch,
            "stability": round(stability, 4),
        })

df_stability = pd.DataFrame(stability_rows)
print("\n=== STABILITY ACROSS SEEDS ===")
print(df_stability.sort_values("stability", ascending=False).to_string(index=False))

In [ ]:
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

def compare_models_stability(results, seeds, probe_head, probe_tail, models):
    fig, axes = plt.subplots(1, len(models), figsize=(18, 5))
    
    colors = ['red', 'blue', 'green']
    
    for j, model_name in enumerate(models):
        ax = axes[j]
        pca = PCA(n_components=2)
        
        for i, seed in enumerate(seeds):
            key = f"{model_name}_s{seed}_e100"
            model = results[key].model
            
            embs = model.entity_representations[0](indices=None).detach().cpu().numpy()
            reduced_embs = pca.fit_transform(embs)
            
            h_idx = entity_to_id[probe_head]
            t_idx = entity_to_id[probe_tail]
            
            ax.scatter(reduced_embs[h_idx, 0], reduced_embs[h_idx, 1], 
                       color=colors[i], marker='o')
            ax.scatter(reduced_embs[t_idx, 0], reduced_embs[t_idx, 1], 
                       color=colors[i], marker='x')
            
            ax.arrow(reduced_embs[h_idx, 0], reduced_embs[h_idx, 1], 
                     reduced_embs[t_idx, 0] - reduced_embs[h_idx, 0], 
                     reduced_embs[t_idx, 1] - reduced_embs[h_idx, 1],
                     color=colors[i], alpha=0.3, head_width=0.05)
        
        ax.set_title(model_name)
        ax.grid(True, linestyle='--', alpha=0.6)

    plt.suptitle(f"Model Stability Comparison: {probe_head} → {probe_tail}")
    plt.show()

In [ ]:
compare_models_stability(results, seeds, probe_head, probe_tail,
                         ["TransH", "ConvKB", "DistMult"])

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
def build_similarity_context(model,triples_factory,top_k=6):
    embs = model.entity_represenattions[0](indices = None).detach().cpu().numpy()
    # calculate the similliriity between the entire row = i and column = j that means just be able to understand the cosine simillarrity bertween every node and its neighbours (14,14)
    sim_matrix = cosine_similarity(embs) 
    